# Time Series Forecasting with Lag Features

A time series is a sequence of measurements ordered in **time** — daily sales, hourly temperature, a stock's closing price. Forecasting means using the past to predict the future.

The trick that makes ordinary regression work on a time series is **turning the series into a supervised table**: for each day we build a row of *features known at or before that day* (yesterday's value, last week's value, a rolling average, the day-of-week) and a *target* (today's value). Once the data is in `(X, y)` shape, any regressor — Ridge, RandomForest — can learn the mapping.

But time series break one assumption ordinary ML relies on: rows are **not** independent and interchangeable. Order matters. That forces two discipline changes:

1. **Never shuffle.** We split by *time* — train on the earlier period, test on the later period — because in reality you only ever have the past to predict the future.
2. **Watch for leakage.** Any feature that peeks at future values (or a random split that scatters future rows into the training set) inflates your score and collapses in production.

We will synthesize a realistic daily series with **trend + seasonality + noise**, engineer lag / rolling / calendar features, train a model with a proper time-based split, and compare it against naive baselines using **MAE** and **RMSE**.

In [ ]:
import numpy as np                                      # numeric arrays and math
import pandas as pd                                    # the time series lives in a DataFrame indexed by date
import matplotlib.pyplot as plt                        # plotting
import seaborn as sns                                  # nicer default styling

from sklearn.ensemble import RandomForestRegressor     # comparison model (nonlinear, but cannot extrapolate a trend)
from sklearn.linear_model import Ridge                 # PRIMARY forecaster: linear, extrapolates the trend cleanly
from sklearn.metrics import mean_absolute_error, mean_squared_error  # evaluation metrics
from sklearn.model_selection import TimeSeriesSplit    # time-aware cross-validation (never shuffles)

# --- Reproducibility: seed everything so every run prints identical numbers ---
SEED = 42
rng = np.random.default_rng(SEED)   # modern NumPy Generator; drives all the synthetic noise below
np.random.seed(SEED)                # legacy global RNG, seeded as a safe default

sns.set_theme(style="whitegrid")    # apply seaborn's clean grid style to all matplotlib plots
print("setup complete")

## 1. Synthesize a daily time series

Real series are usually a sum of a few interpretable pieces. We build ours from a classic **additive decomposition**:

$$y_t = \underbrace{T_t}_{\text{trend}} + \underbrace{S^{(w)}_t}_{\text{weekly}} + \underbrace{S^{(y)}_t}_{\text{yearly}} + \underbrace{\varepsilon_t}_{\text{noise}}$$

- **Trend** $T_t$ — a slow upward drift (e.g. a growing business).
- **Weekly seasonality** $S^{(w)}_t$ — a 7-day cycle (weekends differ from weekdays).
- **Yearly seasonality** $S^{(y)}_t$ — a 365-day cycle (summer vs. winter), modeled with a sine wave.
- **Noise** $\varepsilon_t$ — Gaussian randomness that is *not* predictable; it sets a floor on how good any forecast can be.

We generate **~3 years** of daily data. Knowing the true components lets us sanity-check later that the model recovers the structure and not the noise.

In [ ]:
# --- Build a daily DatetimeIndex spanning three full years ---
n_days = 365 * 3                                              # ~3 years of daily observations
dates = pd.date_range(start="2021-01-01", periods=n_days, freq="D")  # one row per calendar day
t = np.arange(n_days)                                         # integer day counter 0, 1, 2, ... for the trend/seasonal math

# --- Component 1: linear trend -- a slow, steady climb over the whole period ---
trend = 50 + 0.05 * t                                         # starts near 50, rises ~0.05 units/day (~55 over 3 yrs)

# --- Component 2: yearly seasonality -- one full sine cycle every 365 days ---
# amplitude 10 means the yearly swing is +/-10 units (e.g. summer peak, winter trough)
yearly = 10 * np.sin(2 * np.pi * t / 365.25)

# --- Component 3: weekly seasonality -- a repeating 7-day pattern keyed to day-of-week ---
# Hand-pick an offset per weekday (Mon=0 ... Sun=6): sales dip midweek, jump on the weekend.
weekday_effect = np.array([-2, -1, 0, 1, 3, 6, 5])           # index by dates.dayofweek
weekly = weekday_effect[dates.dayofweek.to_numpy()]          # map each date to its weekday offset

# --- Component 4: irreducible noise -- Gaussian jitter the model can never predict ---
noise = rng.normal(loc=0.0, scale=2.0, size=n_days)          # mean 0, std 2

# --- Additive sum of all components = the observed series ---
y_series = trend + yearly + weekly + noise

# Store as a one-column DataFrame INDEXED BY DATE -- the natural container for a time series.
df = pd.DataFrame({"sales": y_series}, index=dates)
df.index.name = "date"

print(df.shape)          # (1095, 1)
print(df.head())         # peek at the first few days

In [ ]:
# Plot the raw series. You should SEE the three ingredients:
#   - an overall upward slope        -> the trend
#   - a slow up-and-down wave        -> the yearly seasonality
#   - a dense weekly wiggle + fuzz   -> the weekly cycle plus noise
plt.figure(figsize=(13, 4))
plt.plot(df.index, df["sales"], linewidth=0.8, color="#3b6ea5")
plt.title("Synthetic daily sales — trend + weekly & yearly seasonality + noise")
plt.xlabel("date"); plt.ylabel("sales")
plt.tight_layout(); plt.show()

# Zoom into ~8 weeks to make the repeating 7-day pattern obvious.
plt.figure(figsize=(13, 3.5))
plt.plot(df.index[:56], df["sales"][:56], marker="o", markersize=3, linewidth=1, color="#c1652f")
plt.title("Zoom: first 8 weeks — notice the weekend bumps repeating every 7 days")
plt.xlabel("date"); plt.ylabel("sales")
plt.tight_layout(); plt.show()

## 2. From a series to a supervised table (feature engineering)

A regressor needs a feature matrix `X` and a target vector `y`. We manufacture `X` **only from information available up to and including the prediction day** — otherwise we leak the future. Three families of features:

**Lag features.** The value of the series shifted back in time: $y_{t-1}$ (yesterday), $y_{t-7}$ (same weekday last week), $y_{t-14}$, ... . Lags are the single most powerful predictor because tomorrow usually looks a lot like today — this is **autocorrelation**. The 7-day lag directly hands the model the weekly cycle.

**Rolling-window features.** Summaries of a trailing window, e.g. the mean of the last 7 days. A rolling mean smooths out noise and captures the local *level* / short-term trend. Crucially we shift it back by one day so the window ends **yesterday** — a window that includes today would leak the answer.

**Calendar / date-part features.** Attributes read straight off the date: day-of-week, month, day-of-year. These let the model learn seasonality explicitly ("Saturdays are high", "summer is high") without needing a long enough lag.

### Why we drop the warm-up rows
Shifting a column by $k$ rows leaves the first $k$ entries undefined (`NaN`) — there is no "7 days ago" for the first week. A rolling mean of window 7 is undefined until 7 days have accumulated. These early rows have incomplete features, so we **drop them**. Filling them (e.g. with 0 or a mean) would feed the model fabricated history and bias training.

In [ ]:
# Work on a copy so the original df stays clean.
feat = df.copy()

# --- LAG FEATURES: y shifted DOWN by k rows, so row t sees the value from t-k ---
# .shift(k) moves data forward in time -> the value that lands on row t came from k days earlier.
lags = [1, 2, 3, 7, 14]                       # yesterday, 2/3 days ago, last week, two weeks ago
for k in lags:
    feat[f"lag_{k}"] = feat["sales"].shift(k)  # e.g. lag_7 on 2021-01-08 holds the 2021-01-01 value

# --- ROLLING-MEAN FEATURES: average of a trailing window, then SHIFTED so it ends YESTERDAY ---
# .shift(1) first guarantees the window never includes today's value (which would be leakage),
# then .rolling(w).mean() averages the previous w days.
for w in [7, 30]:
    feat[f"roll_mean_{w}"] = feat["sales"].shift(1).rolling(window=w).mean()
feat["roll_std_7"] = feat["sales"].shift(1).rolling(window=7).std()  # trailing 7-day volatility

# --- CALENDAR / DATE-PART FEATURES: read directly off the DatetimeIndex (no leakage — the date is known) ---
feat["dayofweek"] = feat.index.dayofweek       # 0 = Monday ... 6 = Sunday  -> captures the weekly cycle
feat["month"] = feat.index.month               # 1..12                      -> captures the yearly cycle
feat["dayofyear"] = feat.index.dayofyear       # 1..366                     -> finer yearly position
feat["is_weekend"] = (feat.index.dayofweek >= 5).astype(int)  # 1 on Sat/Sun, else 0

print("columns after engineering:", list(feat.columns))
# Show the head: the first rows carry NaNs in the lag/rolling columns (no history yet).
print(feat.head(8)[["sales", "lag_1", "lag_7", "roll_mean_7", "roll_mean_30"]])

In [ ]:
# The longest look-back is roll_mean_30 (needs 30 prior days) plus the shift(1) -> ~31 warm-up rows.
# Any row with a NaN feature is incomplete, so drop it. dropna() removes exactly the warm-up block.
n_before = len(feat)
feat = feat.dropna()
n_after = len(feat)
print(f"dropped {n_before - n_after} warm-up rows (incomplete lags/rolling windows); {n_after} usable rows remain")

# Separate the supervised table into features X and target y.
# The target is today's 'sales'; every column in X is knowable strictly before/at prediction time.
feature_cols = [c for c in feat.columns if c != "sales"]
X = feat[feature_cols]
y = feat["sales"]
print("X shape:", X.shape, "| y shape:", y.shape)

## 3. The time-based split — and why shuffling is a trap

For i.i.d. data you'd call `train_test_split(..., shuffle=True)`. **For time series that is a bug.** Two reasons:

1. **It leaks the future into the past.** A random shuffle scatters *later* days into the training set. The model then learns from days that, in the real forecasting task, haven't happened yet. Worse, `lag_1` of a test day and the `sales` of its neighbor are almost the same number — if that neighbor is in train, you're essentially training on the answer. Your test score looks fantastic and then falls apart in production.
2. **It doesn't match the real task.** In deployment you always stand at "today" and predict "tomorrow". The evaluation must mimic that: **fit on the earlier period, score on the strictly later period.**

So we slice chronologically: the first 80% of days is the training set, the last 20% is the test set. No overlap, no shuffle — the test set is entirely in the *future* relative to training.

### Stationarity intuition
Classical models (ARIMA) assume the series is **stationary** — its statistical properties (mean, variance) don't change over time. Our series is *not* stationary: it has a rising trend and seasonal swings, so its mean drifts upward. Lag and calendar features are our workaround: differencing against `lag_1` captures change rather than level, and the calendar columns absorb the seasonal mean shifts — letting a plain regressor cope with a non-stationary series.

In [ ]:
# --- CHRONOLOGICAL split: NO shuffling. Earlier days train, later days test. ---
split_frac = 0.80
split_idx = int(len(X) * split_frac)          # index that separates the first 80% from the last 20%

# iloc slices by POSITION, preserving date order -> train is entirely before test in time.
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

# The last training date is strictly earlier than the first test date -- the essential guarantee.
print(f"train: {X_train.index.min().date()} -> {X_train.index.max().date()}  ({len(X_train)} days)")
print(f"test : {X_test.index.min().date()} -> {X_test.index.max().date()}  ({len(X_test)} days)")
assert X_train.index.max() < X_test.index.min(), "train must end before test begins!"

## 4. Train a model and forecast the test period

Our primary forecaster is **`Ridge`** — linear regression with a small L2 penalty. Why linear here? The series has a **rising trend**, and the test period sits at *higher* sales levels than anything in training. A linear model reads the trend off the lag/rolling features and extrapolates it cleanly.

A **`RandomForestRegressor`** (an ensemble of decision trees) is trained too, as a comparison. But note a structural limitation: **tree models cannot extrapolate** — every prediction is an average of training-set targets, so a forest can never output a value above the highest level it saw in training. On a trending series that caps its accuracy, and we'll see it lose to the linear model below. (This is a genuinely useful lesson: match the model to the shape of the data.)

The model produces a **one-step-ahead** forecast for each test day: given that day's genuine lag/rolling/calendar features, predict its sales. We evaluate with two error metrics:

$$\text{MAE} = \frac{1}{n}\sum_{i=1}^{n} \lvert y_i - \hat y_i\rvert \qquad\qquad \text{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat y_i)^2}$$

- **MAE** is the average absolute miss, in the same units as sales — easy to read.
- **RMSE** squares the errors first, so it punishes large misses more heavily. RMSE $\geq$ MAE always; a big gap between them signals occasional large errors.

In [ ]:
def report(name, y_true, y_hat):
    """Print MAE and RMSE for one set of predictions and return them as a tuple."""
    mae = mean_absolute_error(y_true, y_hat)                 # average absolute error
    rmse = np.sqrt(mean_squared_error(y_true, y_hat))        # sqrt of mean squared error (units of sales)
    print(f"{name:<22} MAE = {mae:6.3f}   RMSE = {rmse:6.3f}")
    return mae, rmse


# --- PRIMARY model: Ridge, fit on the training slice only ---
ridge = Ridge(alpha=1.0)          # small L2 penalty; keeps coefficients stable
ridge.fit(X_train, y_train)       # learn features -> sales on the PAST
pred_ridge = ridge.predict(X_test)  # one-step-ahead forecast for every FUTURE test day

# --- COMPARISON model: RandomForest (will underperform because it can't extrapolate the trend) ---
rf = RandomForestRegressor(
    n_estimators=300,     # number of trees; more = steadier predictions (still fast on ~850 rows)
    max_depth=None,       # let trees grow fully; the ensemble average controls variance
    random_state=SEED,    # seed the bootstrap sampling + feature selection -> reproducible
    n_jobs=-1,            # use all CPU cores to train trees in parallel
)
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)

print("--- Model performance on the held-out FUTURE test period ---")
mae_ridge, rmse_ridge = report("Ridge (primary)", y_test, pred_ridge)
mae_rf, rmse_rf = report("RandomForest", y_test, pred_rf)

## 5. Naive baselines — is the model actually earning its keep?

A forecast is only impressive relative to the trivial alternative. Two standard baselines:

- **Naive (persistence):** predict tomorrow = today, i.e. $\hat y_t = y_{t-1}$. Surprisingly hard to beat for many series.
- **Seasonal naive:** predict this day = the same day one season ago, i.e. $\hat y_t = y_{t-7}$ for weekly seasonality. This baseline already "knows" the weekly cycle.

Both are just columns we already engineered (`lag_1` and `lag_7`), so we get them for free. **If our model can't beat these, the features or model aren't adding value.**

In [ ]:
# The baselines are literally the lag columns of the TEST rows -- no fitting required.
pred_naive = X_test["lag_1"].to_numpy()        # persistence: yesterday's value
pred_snaive = X_test["lag_7"].to_numpy()       # seasonal naive: value 7 days ago (same weekday)

print("--- Naive baselines on the same test period ---")
mae_naive, rmse_naive = report("Naive (t-1)", y_test, pred_naive)
mae_snaive, rmse_snaive = report("Seasonal naive (t-7)", y_test, pred_snaive)

print("\n--- Head-to-head (lower is better) ---")
report("Ridge (primary)", y_test, pred_ridge)
report("RandomForest", y_test, pred_rf)

# Quantify how much the PRIMARY model improves on the BEST naive baseline.
best_naive_mae = min(mae_naive, mae_snaive)
improve = 100 * (best_naive_mae - mae_ridge) / best_naive_mae
print(f"\nRidge MAE is {improve:.1f}% lower than the best naive baseline.")
print("(RandomForest lands above the seasonal-naive baseline -- it cannot extrapolate the rising trend.)")

In [ ]:
# --- Actual vs. predicted over the test dates (primary model = Ridge) ---
plt.figure(figsize=(13, 4.5))
plt.plot(y_test.index, y_test.values, label="actual", color="#222222", linewidth=1.3)
plt.plot(y_test.index, pred_ridge, label="Ridge forecast", color="#c1652f", linewidth=1.1)
plt.plot(y_test.index, pred_naive, label="naive (t-1)", color="#8aa0b6", linewidth=0.8, alpha=0.7)
plt.title("Forecast vs. actual on the held-out future period")
plt.xlabel("date"); plt.ylabel("sales"); plt.legend()
plt.tight_layout(); plt.show()

# --- Which features did the RandomForest rely on? (impurity-based importances) ---
# We read importances from the forest because they are directly comparable across features;
# raw Ridge coefficients would depend on each feature's scale and mislead here.
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values()
plt.figure(figsize=(8, 4.5))
importances.plot(kind="barh", color="#3b6ea5")
plt.title("RandomForest feature importances")
plt.xlabel("importance")
plt.tight_layout(); plt.show()

## 6. Time-aware cross-validation with `TimeSeriesSplit`

A single train/test split uses one slice of history for evaluation — noisy if that slice happens to be easy or hard. Ordinary k-fold CV would shuffle folds and leak the future, so scikit-learn provides **`TimeSeriesSplit`**: an expanding-window scheme where every fold's training set is entirely *before* its validation set.

```
fold 1:  train [====]              test [==]
fold 2:  train [======]            test [==]
fold 3:  train [========]          test [==]
```

The training window grows and always precedes the validation window in time — the same no-leakage discipline as our main split, repeated across several cut points for a more robust estimate. We cross-validate the primary model (Ridge).

In [ ]:
# 5 expanding-window folds. Each fold trains on everything before its validation block.
tscv = TimeSeriesSplit(n_splits=5)

cv_maes = []
for fold, (tr_idx, va_idx) in enumerate(tscv.split(X), start=1):
    # tr_idx / va_idx are POSITIONAL indices; because we don't shuffle, tr_idx always precedes va_idx.
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

    model = Ridge(alpha=1.0)
    model.fit(X_tr, y_tr)
    mae = mean_absolute_error(y_va, model.predict(X_va))
    cv_maes.append(mae)
    # Show that each fold's train window ends before its validation window begins.
    print(f"fold {fold}: train {len(tr_idx):4d} days (up to {X_tr.index.max().date()}) | "
          f"valid {len(va_idx)} days (from {X_va.index.min().date()}) | MAE = {mae:.3f}")

print(f"\nmean CV MAE = {np.mean(cv_maes):.3f}  (+/- {np.std(cv_maes):.3f})")
print("(Fold 1 has the least history to learn the trend, so its error is the largest.)")

## 7. Seeing the leakage bug for real

Talk is cheap — let's *measure* the damage. We take one model (the RandomForest, whose leakage sensitivity is easy to see) and evaluate it **two ways**: with our honest time-based split, and with a **random shuffled split** (the wrong way). Because consecutive days are nearly identical and `lag_1` of a test day is essentially a neighbor's target, scattering neighbors into training lets the model "cheat". The shuffled test error comes out **misleadingly low** — a score you could never reproduce when actually forecasting the future.

In [ ]:
from sklearn.model_selection import train_test_split  # imported here only to demonstrate the WRONG approach

# WRONG: shuffle=True mixes future days into the training set (leakage).
Xtr_bad, Xte_bad, ytr_bad, yte_bad = train_test_split(
    X, y, test_size=0.20, shuffle=True, random_state=SEED
)
# Same model type both ways, so the ONLY difference is how we split -> isolates the leakage effect.
bad = RandomForestRegressor(n_estimators=300, random_state=SEED, n_jobs=-1).fit(Xtr_bad, ytr_bad)
mae_leaky = mean_absolute_error(yte_bad, bad.predict(Xte_bad))

print(f"Shuffled split   MAE = {mae_leaky:6.3f}   <- looks better, but it's LEAKED / not achievable")
print(f"Time-based split MAE = {mae_rf:6.3f}   <- honest estimate of future performance")
print("\nThe shuffled score is optimistically low: it trained on days from the future.")
print("Always trust the time-based number.")

## Summary

- A time series is **trend + seasonality + noise**; forecasting recovers the predictable structure while the noise sets the error floor.
- The core move is turning the series into a **supervised table** via **lag**, **rolling-window**, and **calendar** features — all computed from information available at prediction time.
- **Drop the warm-up rows**: early lags/rolling windows are undefined, and faking them biases the model.
- **Match the model to the data.** With a rising trend, linear `Ridge` extrapolates and wins; a `RandomForest` cannot predict beyond values seen in training, so it lags behind on a trending series.
- **Split by time, never shuffle.** A random split leaks the future into the past, producing an inflated score that collapses in production. Use `TimeSeriesSplit` for time-aware cross-validation.
- Judge the model against **naive baselines** (persistence `t-1`, seasonal `t-7`) using **MAE** and **RMSE** — beating them is the bar for adding value.